In [ ]:
# ── CELL 1: installs ──────────────────────────────────────────────────────────
# Kaggle already has torch+cu121, numpy 1.x, transformers — minimal installs needed
import subprocess, sys

pkgs = ["einops", "wandb", "tqdm"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Installs done.")

In [ ]:
# ── CELL 2: imports + device ──────────────────────────────────────────────────
import torch
import torch.nn as nn
import numpy as np
from transformers import GPT2Model, GPT2Config
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os, math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"numpy:        {np.__version__}")
print(f"torch:        {torch.__version__}")
import transformers as _tr
print(f"transformers: {_tr.__version__}")
print(f"device:       {device}")
if device.type == "cuda":
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ── CELL 3: DecisionTransformer model ─────────────────────────────────────────
class DecisionTransformer(nn.Module):
    """
    GPT-2 backbone conditioned on (RTG, state, action) token triples.
    Predicts the next action from the state token output position.
    """
    def __init__(
        self,
        state_dim:   int   = 128,   # CNN output dimension
        act_dim:     int   = 6,     # max actions across all games
        hidden_size: int   = 128,
        context_len: int   = 30,    # K — timesteps per window
        n_layer:     int   = 6,
        n_head:      int   = 8,
        dropout:     float = 0.1,
        in_channels: int   = 4,     # stacked grayscale frames
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.context_len = context_len
        self.act_dim     = act_dim

        # CNN encoder — matches DQN architecture from original DT paper
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),  # (32,20,20)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),           # (64, 9, 9)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),           # (64, 7, 7)
            nn.ReLU(),
            nn.Flatten(),                                          # 3136
            nn.Linear(3136, state_dim),
            nn.ReLU(),
        )

        # Token embeddings — all projected to hidden_size
        self.embed_state    = nn.Linear(state_dim, hidden_size)
        self.embed_action   = nn.Embedding(act_dim, hidden_size)
        self.embed_rtg      = nn.Linear(1, hidden_size)
        self.embed_timestep = nn.Embedding(10000, hidden_size)
        self.embed_ln       = nn.LayerNorm(hidden_size)

        # GPT-2 backbone
        gpt_cfg = GPT2Config(
            vocab_size    = 1,               # unused — we bypass token embeddings
            n_embd        = hidden_size,
            n_layer       = n_layer,
            n_head        = n_head,
            n_inner       = 4 * hidden_size,
            n_positions   = 3 * context_len, # 3 tokens per timestep
            resid_pdrop   = dropout,
            attn_pdrop    = dropout,
            embd_pdrop    = dropout,
        )
        self.transformer    = GPT2Model(gpt_cfg)
        self.predict_action = nn.Linear(hidden_size, act_dim)

    def forward(self, states, actions, rtg, timesteps, attn_mask=None):
        """
        states:    (B, K, 4, 84, 84)
        actions:   (B, K)
        rtg:       (B, K, 1)
        timesteps: (B, K)
        returns:   (B, K, act_dim) action logits
        """
        B, K = states.shape[:2]

        # Encode frames through CNN
        state_emb = self.cnn(
            states.view(B * K, *states.shape[2:])
        ).view(B, K, -1)                                    # (B, K, state_dim)

        # Project + add timestep embedding to each token type
        time_emb   = self.embed_timestep(timesteps)         # (B, K, H)
        state_emb  = self.embed_state(state_emb)  + time_emb
        action_emb = self.embed_action(actions)    + time_emb
        rtg_emb    = self.embed_rtg(rtg)           + time_emb

        # Interleave: [RTG_0, s_0, a_0, RTG_1, s_1, a_1, ...]
        tokens = (
            torch.stack([rtg_emb, state_emb, action_emb], dim=2)  # (B, K, 3, H)
            .view(B, 3 * K, self.hidden_size)                      # (B, 3K, H)
        )
        tokens = self.embed_ln(tokens)

        if attn_mask is not None:
            attn_mask = attn_mask.repeat_interleave(3, dim=1)      # (B, 3K)

        x = self.transformer(
            inputs_embeds=tokens, attention_mask=attn_mask
        ).last_hidden_state                                         # (B, 3K, H)

        # Extract state-token outputs (index 1 in each triple) → predict action
        x = x.view(B, K, 3, self.hidden_size)[:, :, 1, :]         # (B, K, H)
        return self.predict_action(x)                               # (B, K, act_dim)

print("DecisionTransformer defined.")

In [ ]:
# ── CELL 4: Dataset ────────────────────────────────────────────────────────────
class DecisionTransformerDataset(Dataset):
    """
    Sliding-window dataset over a list of episodes.
    Each window: K timesteps of (obs, action, rtg, timestep_index).
    Windows never cross episode boundaries.
    """
    def __init__(self, episodes, context_len=30, stride=15, gamma=1.0):
        self.K       = context_len
        self.windows = []

        for ep in episodes:
            obs     = ep["obs"]      # (T, 4, 84, 84) float32 in [0,1]
            actions = ep["actions"]  # (T,) int64
            rewards = ep["rewards"]  # (T,) float32
            T = len(actions)

            if T < context_len:
                continue

            # RTG: undiscounted sum of future rewards within this episode
            rtg     = np.zeros(T, dtype=np.float32)
            running = 0.0
            for t in reversed(range(T)):
                running += rewards[t] * (gamma ** 0)  # gamma=1 for DT
                rtg[t]   = running

            for start in range(0, T - context_len + 1, stride):
                end = start + context_len
                self.windows.append((
                    obs    [start:end],            # (K, 4, 84, 84)
                    actions[start:end],            # (K,)
                    rtg    [start:end],            # (K,)
                    np.arange(start, end, dtype=np.int64),  # (K,) abs timesteps
                ))

        print(f"  {len(self.windows):,} windows ",
              f"from {len(episodes)} episodes ",
              f"(K={context_len}, stride={stride})")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        obs, acts, rtg, ts = self.windows[idx]
        return {
            "obs":       torch.tensor(obs,  dtype=torch.float32),   # (K,4,84,84)
            "acts":      torch.tensor(acts, dtype=torch.long),       # (K,)
            "rtg":       torch.tensor(rtg,  dtype=torch.float32).unsqueeze(-1),  # (K,1)
            "timesteps": torch.tensor(ts,   dtype=torch.long),       # (K,)
            "mask":      torch.ones(self.K, dtype=torch.long),        # (K,)
        }

print("DecisionTransformerDataset defined.")

In [ ]:
# ── CELL 5: synthetic data + loaders ──────────────────────────────────────────
# Realistic synthetic episodes — replace with real D4RL data once pipeline works

GAME_CONFIGS = {
    "Breakout": {"act_dim": 4,  "target_rtg": 90.0},
    "Pong":     {"act_dim": 6,  "target_rtg": 20.0},
    "Qbert":    {"act_dim": 6,  "target_rtg": 2500.0},
}

def make_synthetic_episodes(n_episodes=100, min_len=100, max_len=300, act_dim=4):
    """
    Generates episodes with the correct array shapes and dtypes.
    Rewards are sparse (20% of steps) to mimic Atari reward structure.
    """
    rng = np.random.default_rng(seed=42)
    episodes = []
    for _ in range(n_episodes):
        T = rng.integers(min_len, max_len)
        episodes.append({
            "obs":     rng.random((T, 4, 84, 84), dtype=np.float32),
            "actions": rng.integers(0, act_dim, T).astype(np.int64),
            "rewards": (rng.random(T) * (rng.random(T) > 0.8)).astype(np.float32),
        })
    return episodes

print("Building dataloaders...")
loaders = {}
for game, cfg in GAME_CONFIGS.items():
    eps = make_synthetic_episodes(
        n_episodes=100,
        min_len=100, max_len=300,
        act_dim=cfg["act_dim"],
    )
    ds = DecisionTransformerDataset(eps, context_len=30, stride=15)
    loaders[game] = DataLoader(
        ds,
        batch_size  = 64,    # T4 has 16GB VRAM — 64 is fine
        shuffle     = True,
        num_workers = 2,     # Kaggle allows multiprocessing
        pin_memory  = True,  # faster CPU→GPU transfer
        drop_last   = True,
    )
    b = next(iter(loaders[game]))
    print(f"  {game}: obs={b['obs'].shape} acts={b['acts'].shape} rtg={b['rtg'].shape}")

print("Loaders ready.")

In [ ]:
# ── CELL 6: instantiate model ─────────────────────────────────────────────────
model = DecisionTransformer(
    state_dim   = 128,
    act_dim     = 6,     # 6 covers all three games (Breakout only uses 4)
    hidden_size = 128,
    context_len = 30,
    n_layer     = 6,
    n_head      = 8,
    dropout     = 0.1,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params/1e6:.2f}M")
print(f"Device:     {next(model.parameters()).device}")

# Verify one forward pass before training
b = next(iter(loaders["Breakout"]))
with torch.no_grad():
    logits = model(
        b["obs"]      .to(device),
        b["acts"]     .to(device),
        b["rtg"]      .to(device),
        b["timesteps"].to(device),
        b["mask"]     .to(device),
    )
print(f"Forward pass OK — logits: {logits.shape}")  # (64, 30, 6)

In [ ]:
# ── CELL 7: trainer ───────────────────────────────────────────────────────────
class DecisionTransformerTrainer:
    def __init__(
        self,
        model,
        loaders,
        device,
        lr           = 6e-4,
        weight_decay = 1e-4,
        warmup_steps = 2500,
        max_steps    = 100_000,
        grad_clip    = 0.25,
        log_every    = 100,
        ckpt_every   = 2500,
        ckpt_dir     = "/kaggle/working",
    ):
        self.model      = model
        self.loaders    = loaders
        self.device     = device
        self.grad_clip  = grad_clip
        self.log_every  = log_every
        self.ckpt_every = ckpt_every
        self.ckpt_dir   = ckpt_dir
        self.max_steps  = max_steps
        self.criterion  = nn.CrossEntropyLoss()

        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr           = lr,
            weight_decay = weight_decay,
            betas        = (0.9, 0.95),  # DT paper values
        )

        # Linear warmup → cosine decay
        def lr_lambda(step):
            if step < warmup_steps:
                return step / max(1, warmup_steps)
            progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
            return 0.5 * (1.0 + math.cos(math.pi * progress))

        self.scheduler = torch.optim.lr_scheduler.LambdaLR(
            self.optimizer, lr_lambda
        )

        # Infinite iterators — one per game
        self.iters = {g: self._cycle(l) for g, l in loaders.items()}
        self.games = list(loaders.keys())

    @staticmethod
    def _cycle(loader):
        while True:
            for batch in loader:
                yield batch

    def _step(self, batch):
        obs       = batch["obs"]      .to(self.device)
        acts      = batch["acts"]     .to(self.device)
        rtg       = batch["rtg"]      .to(self.device)
        timesteps = batch["timesteps"].to(self.device)
        mask      = batch["mask"]     .to(self.device)

        logits = self.model(obs, acts, rtg, timesteps, mask)  # (B, K, act_dim)

        # Only compute loss on real (non-padded) tokens
        flat_logits = logits.view(-1, logits.shape[-1])
        flat_acts   = acts.view(-1)
        flat_mask   = mask.view(-1).bool()

        loss = self.criterion(flat_logits[flat_mask], flat_acts[flat_mask])

        with torch.no_grad():
            acc = (
                flat_logits[flat_mask].argmax(-1) == flat_acts[flat_mask]
            ).float().mean().item()

        return loss, acc

    def train(self):
        self.model.train()
        best_loss = float("inf")
        history   = {"step": [], "loss": [], "acc": [], "lr": []}
        pbar      = tqdm(total=self.max_steps, desc="Training", dynamic_ncols=True)

        for step in range(1, self.max_steps + 1):
            game  = self.games[step % len(self.games)]
            batch = next(self.iters[game])

            self.optimizer.zero_grad()
            loss, acc = self._step(batch)
            loss.backward()
            grad_norm = nn.utils.clip_grad_norm_(
                self.model.parameters(), self.grad_clip
            )
            self.optimizer.step()
            self.scheduler.step()
            pbar.update(1)

            if step % self.log_every == 0:
                lr_now = self.scheduler.get_last_lr()[0]
                pbar.set_postfix({
                    "game":  game[:3],
                    "loss":  f"{loss.item():.4f}",
                    "acc":   f"{acc:.3f}",
                    "lr":    f"{lr_now:.2e}",
                    "gnorm": f"{grad_norm:.3f}",
                })
                history["step"].append(step)
                history["loss"].append(loss.item())
                history["acc"].append(acc)
                history["lr"].append(lr_now)

            if step % self.ckpt_every == 0:
                if loss.item() < best_loss:
                    best_loss = loss.item()
                    path = os.path.join(
                        self.ckpt_dir,
                        f"dt_step{step:06d}_loss{loss.item():.4f}.pt"
                    )
                    torch.save({
                        "step":       step,
                        "loss":       loss.item(),
                        "model":      self.model.state_dict(),
                        "optimizer":  self.optimizer.state_dict(),
                        "scheduler":  self.scheduler.state_dict(),
                    }, path)
                    tqdm.write(f"  ✓ checkpoint saved: {path}")

        pbar.close()
        print(f"
Training complete. Best loss: {best_loss:.4f}")
        return history

print("DecisionTransformerTrainer defined.")

In [ ]:
# ── CELL 8: train ─────────────────────────────────────────────────────────────
trainer = DecisionTransformerTrainer(
    model        = model,
    loaders      = loaders,
    device       = device,
    lr           = 6e-4,
    weight_decay = 1e-4,
    warmup_steps = 2500,
    max_steps    = 100_000,
    grad_clip    = 0.25,
    log_every    = 100,
    ckpt_every   = 2500,
    ckpt_dir     = "/kaggle/working",
)

history = trainer.train()

In [ ]:
# ── CELL 9: loss curve ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history["step"], history["loss"], lw=1.5, color="#534AB7")
ax1.set_xlabel("Step"); ax1.set_ylabel("Cross-entropy loss")
ax1.set_title("Training loss")
ax1.axhline(y=1.386, ls="--", color="gray", lw=0.8, label="random baseline (log 4)")
ax1.legend()

ax2.plot(history["step"], history["acc"], lw=1.5, color="#1D9E75")
ax2.set_xlabel("Step"); ax2.set_ylabel("Action accuracy")
ax2.set_title("Training accuracy")
ax2.axhline(y=0.25, ls="--", color="gray", lw=0.8, label="random baseline")
ax2.legend()

plt.tight_layout()
plt.savefig("/kaggle/working/training_curves.png", dpi=150)
plt.show()
print("Saved: /kaggle/working/training_curves.png")